# UPI App Decline Analysis
## Notebook 02 - SQL Runner
---
**What this notebook does (and ONLY this):**
- Connects Python to SQL Server (`localhost\SQLEXPRESS`)
- Loads each `.sql` file from the `sql/` folder
- Splits on `GO` and runs each batch individually
- Displays results as clean pandas DataFrames
- Saves each result set as a CSV to `sql_results/` for Notebook 03

**What this notebook does NOT do:** No charts. No analysis. No visualisations.

---
**Folder structure required:**
```
your_project/
├── 02_sql_runner.ipynb
├── sql/
│   ├── query1_ratings_overview.sql
│   ├── query2_ratings_trend_by_year.sql
│   ├── query3_complaint_categories.sql
│   ├── query4_paytm_decline_deepdive.sql
│   ├── query5_company_response_analysis.sql
│   └── query6_era_comparison.sql
└── sql_results/     <- created automatically by Step 1
```


## Step 1 - Imports and Setup

In [1]:
import pyodbc
import pandas as pd
import os
import re

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

os.makedirs('sql_results', exist_ok=True)

print('Imports done')
print(f'  pyodbc  : {pyodbc.version}')
print(f'  pandas  : {pd.__version__}')
print(f'  Output  : sql_results/ folder ready')

Imports done
  pyodbc  : 5.2.0
  pandas  : 2.2.3
  Output  : sql_results/ folder ready


## Step 2 - Connect to SQL Server
Using Windows Authentication - no username or password needed.


In [2]:
SERVER   = r'localhost\SQLEXPRESS'
DATABASE = 'UPI_Analysis'

CONNECTION_STRING = (
    f'DRIVER={{ODBC Driver 17 for SQL Server}};'
    f'SERVER={SERVER};'
    f'DATABASE={DATABASE};'
    f'Trusted_Connection=yes;'
)

try:
    conn = pyodbc.connect(CONNECTION_STRING)
    cursor = conn.cursor()
    cursor.execute('SELECT @@VERSION')
    version = cursor.fetchone()[0].split('\n')[0]
    cursor.execute('SELECT COUNT(*) FROM dbo.reviews')
    count = cursor.fetchone()[0]
    print('Connected successfully')
    print(f'  Server   : {SERVER}')
    print(f'  Database : {DATABASE}')
    print(f'  Version  : {version}')
    print(f'  Rows     : {count:,}')
except Exception as e:
    print(f'Connection failed: {e}')

Connected successfully
  Server   : localhost\SQLEXPRESS
  Database : UPI_Analysis
  Version  : Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
  Rows     : 189,634


## Step 3 - SQL Runner Helper Function

**What it does:**
1. Loads the `.sql` file
2. Splits on `GO` into individual batches
3. Skips empty, comment-only, and USE statement batches
4. Executes each batch and collects result sets as DataFrames
5. Saves each result set as a CSV

**Key design decision:**
Comment stripping is used ONLY for the empty-check, not for execution.
The original batch with comments intact is always what gets executed.
This prevents comment-heavy headers from being skipped incorrectly.


In [3]:
def run_sql_file(filepath, conn, save_prefix=None):
    with open(filepath, 'r', encoding='utf-8') as f:
        raw_sql = f.read()

    batches = re.split(
        r'^\s*GO\s*$', raw_sql,
        flags=re.MULTILINE | re.IGNORECASE
    )

    results    = []
    result_num = 0

    for i, batch in enumerate(batches):
        stripped = batch.strip()
        if not stripped:
            continue

        # Strip comments only for skip-check
        no_comments = re.sub(r'--[^\n]*', '', stripped).strip()
        if not no_comments:
            continue
        if no_comments.upper().startswith('USE '):
            continue

        # Always execute original batch with comments intact
        try:
            cursor = conn.cursor()
            cursor.execute(stripped)

            if cursor.description:
                cols = [col[0] for col in cursor.description]
                rows = cursor.fetchall()
                df   = pd.DataFrame.from_records(rows, columns=cols)
                results.append(df)

                if save_prefix:
                    result_num += 1
                    csv_path = f'sql_results/{save_prefix}_result{result_num}.csv'
                    df.to_csv(csv_path, index=False)
                    print(f'  Saved : {csv_path}  ({len(df)} rows)')

        except Exception as e:
            print(f'  Batch {i+1} error : {e}')
            print(f'  Preview          : {stripped[:120]}')

    return results

print('Helper function ready.')

Helper function ready.


## Step 4 - Query 1: Ratings Overview

**Business question:** What is the overall rating picture for each app?

**Expected:** 1 result set - one row per app


In [4]:
print('Running: query1_ratings_overview.sql')
print('-' * 50)

q1 = run_sql_file('sql/query1_ratings_overview.sql', conn, 'q1_ratings_overview')

print()
print('=== Q1: Ratings Overview ===')
print('avg rating, negative %, positive % per app')
print()
display(q1[0])

Running: query1_ratings_overview.sql
--------------------------------------------------
  Saved : sql_results/q1_ratings_overview_result1.csv  (3 rows)

=== Q1: Ratings Overview ===
avg rating, negative %, positive % per app



,app,total_reviews,avg_rating,negative_count,neutral_count,positive_count,negative_pct,positive_pct
0,GPay,64554,2.59,36729,4609,23216,56.900000000000,36.000000000000
1,Paytm,62191,2.90,30811,3105,28275,49.500000000000,45.500000000000
2,PhonePe,62889,3.17,26062,4008,32819,41.400000000000,52.200000000000


## Step 5 - Query 2: Rating Trend by Year

**Business question:** Have ratings been getting worse over time?

**Expected:** 1 result set - one row per app per year


In [5]:
print('Running: query2_ratings_trend_by_year.sql')
print('-' * 50)

q2 = run_sql_file('sql/query2_ratings_trend_by_year.sql', conn, 'q2_ratings_trend')

print()
print('=== Q2: Rating Trend by Year ===')
print('avg rating per year with LAG() comparison and direction label')
print()
display(q2[0])

Running: query2_ratings_trend_by_year.sql
--------------------------------------------------
  Saved : sql_results/q2_ratings_trend_result1.csv  (21 rows)

=== Q2: Rating Trend by Year ===
avg rating per year with LAG() comparison and direction label



,app,year,review_count,avg_rating,prev_year_rating,rating_change,direction
0,GPay,2018,1252,2.40,NaN,NaN,BASELINE
1,GPay,2019,7229,1.78,2.40,-0.62,DECLINED
2,GPay,2020,10887,1.61,1.78,-0.17,DECLINED
3,GPay,2021,8631,1.48,1.61,-0.13,DECLINED
4,GPay,2022,3427,1.62,1.48,0.14,IMPROVED
5,GPay,2023,31640,3.48,1.62,1.86,IMPROVED
6,GPay,2026,1488,3.65,3.48,0.17,IMPROVED
7,Paytm,2018,1973,1.68,NaN,NaN,BASELINE
8,Paytm,2019,6336,2.28,1.68,0.60,IMPROVED
9,Paytm,2020,6402,1.92,2.28,-0.36,DECLINED


## Step 6 - Query 3: Complaint Categories

**Business question:** What are users actually complaining about?

**Expected:** 1 result set - categories per app ranked by volume


In [6]:
print('Running: query3_complaint_categories.sql')
print('-' * 50)

q3 = run_sql_file('sql/query3_complaint_categories.sql', conn, 'q3_complaint_categories')

print()
print('=== Q3: Complaint Categories ===')
print('CASE WHEN keyword classification ranked per app')
print()
display(q3[0])

Running: query3_complaint_categories.sql
--------------------------------------------------
  Saved : sql_results/q3_complaint_categories_result1.csv  (21 rows)

=== Q3: Complaint Categories ===
CASE WHEN keyword classification ranked per app



,app,complaint_category,complaint_count,pct_of_app_negatives,rank_within_app
0,GPay,Other,16686,46.600000000000,1
1,GPay,Technical Failures,7004,19.600000000000,2
2,GPay,Payment Failures,5563,15.600000000000,3
3,GPay,UX Problems,3062,8.600000000000,4
4,GPay,Customer Support,2134,6.000000000000,5
5,GPay,Trust & Security,1256,3.500000000000,6
6,GPay,Regulatory Issues,64,0.200000000000,7
7,Paytm,Other,12182,40.500000000000,1
8,Paytm,Technical Failures,5364,17.800000000000,2
9,Paytm,Customer Support,3540,11.800000000000,3


## Step 7 - Query 4: Paytm Decline Deep Dive

**Business question:** How did Paytm complaints evolve month by month vs PhonePe?

**Expected:** 1 result set - one row per app per month


In [7]:
print('Running: query4_paytm_decline_deepdive.sql')
print('-' * 50)

q4 = run_sql_file('sql/query4_paytm_decline_deepdive.sql', conn, 'q4_paytm_deepdive')

print()
print('=== Q4: Paytm Decline Deep Dive ===')
print('Monthly breakdown for Paytm vs PhonePe with running total')
print()
display(q4[0])

Running: query4_paytm_decline_deepdive.sql
--------------------------------------------------
  Saved : sql_results/q4_paytm_deepdive_result1.csv  (120 rows)

=== Q4: Paytm Decline Deep Dive ===
Monthly breakdown for Paytm vs PhonePe with running total



,app,year_month,total_negative,payment_failures,trust_security,technical_failures,customer_support,regulatory_issues,ux_problems,other_complaints,running_total_negative
0,Paytm,2018-09,265,45,16,54,50,22,18,60,265
1,Paytm,2018-10,549,99,39,103,94,52,40,122,814
2,Paytm,2018-11,429,65,27,84,73,51,29,100,1243
3,Paytm,2018-12,357,59,30,68,43,58,15,84,1600
4,Paytm,2019-01,265,50,15,40,42,16,24,78,1865
...,...,...,...,...,...,...,...,...,...,...,...
115,PhonePe,2023-04,665,87,18,116,37,1,70,336,20422
116,PhonePe,2023-05,1117,69,32,193,44,1,95,683,21539
117,PhonePe,2023-06,2186,117,59,198,55,2,180,1575,23725
118,PhonePe,2023-07,1340,54,44,148,33,9,91,961,25065


## Step 8 - Query 5: Company Response Analysis

**Business question:** Do apps respond to negative reviews? Does it help?

**Note:** Only Kaggle data has reply_text - fresh scrape filtered out.

**Expected:** 2 result sets - overall stats and by year


In [8]:
print('Running: query5_company_response_analysis.sql')
print('-' * 50)

q5 = run_sql_file('sql/query5_company_response_analysis.sql', conn, 'q5_response_analysis')

print()
print('=== Q5 PART A: Response Rate Overview ===')
print('response rate, avg days to reply, rating impact per app')
print()
display(q5[0])

print()
print('=== Q5 PART B: Response Rate by Year ===')
print('did response rates change during the 2023 crisis?')
print()
display(q5[1])

Running: query5_company_response_analysis.sql
--------------------------------------------------
  Saved : sql_results/q5_response_analysis_result1.csv  (3 rows)
  Saved : sql_results/q5_response_analysis_result2.csv  (18 rows)

=== Q5 PART A: Response Rate Overview ===
response rate, avg days to reply, rating impact per app



,app,total_reviews,replied_count,response_rate_pct,avg_rating_all,avg_rating_replied,avg_rating_not_replied,reply_impact_on_rating,avg_days_to_reply,min_days_to_reply,max_days_to_reply,response_rank
0,Paytm,60878,46200,75.900000000000,2.87,2.67,3.52,-0.85,0.20,0,37,1
1,PhonePe,61565,33025,53.600000000000,3.14,2.03,4.43,-2.40,0.90,0,121,2
2,GPay,63066,25080,39.800000000000,2.57,1.40,3.34,-1.94,0.70,0,61,3



=== Q5 PART B: Response Rate by Year ===
did response rates change during the 2023 crisis?



,app,year,total_reviews,replied_count,response_rate_pct,response_rate_change
0,GPay,2018,1252,505,40.300000000000,None
1,GPay,2019,7229,3791,52.400000000000,12.100000000000
2,GPay,2020,10887,6847,62.900000000000,10.400000000000
3,GPay,2021,8631,6174,71.500000000000,8.600000000000
4,GPay,2022,3427,2042,59.600000000000,-11.900000000000
5,GPay,2023,31640,5721,18.100000000000,-41.500000000000
6,Paytm,2018,1973,1249,63.300000000000,None
7,Paytm,2019,6336,3890,61.400000000000,-1.900000000000
8,Paytm,2020,6402,4585,71.600000000000,10.200000000000
9,Paytm,2021,5028,3579,71.200000000000,-0.400000000000


## Step 9 - Query 6: Era Comparison

**Business question:** How did sentiment change across Early, Crisis, Recovery eras?

**Expected:** 3 result sets - sentiment overview, full breakdown, executive summary

**Note on Part C:** The executive summary excludes 'Other' (general dissatisfaction)
before ranking, so the top complaint is always a specific named type.
Part B retains 'Other' for the complete picture.


In [9]:
print('Running: query6_era_comparison.sql')
print('-' * 50)

q6 = run_sql_file('sql/query6_era_comparison.sql', conn, 'q6_era_comparison')

print()
print('=== Q6 PART A: Era Sentiment Overview ===')
print('avg rating, negative %, positive % per app per era')
print()
display(q6[0])

print()
print('=== Q6 PART B: Complaint Breakdown by Era ===')
print('all 7 categories per app per era including Other')
print()
display(q6[1])

print()
print('=== Q6 PART C: Executive Summary ===')
print('9 rows — top specific complaint per app per era')
print()
display(q6[2])

Running: query6_era_comparison.sql
--------------------------------------------------
  Saved : sql_results/q6_era_comparison_result1.csv  (9 rows)
  Saved : sql_results/q6_era_comparison_result2.csv  (63 rows)
  Saved : sql_results/q6_era_comparison_result3.csv  (9 rows)

=== Q6 PART A: Era Sentiment Overview ===
avg rating, negative %, positive % per app per era



,app,era,total_reviews,avg_rating,negative_count,negative_pct,positive_count,positive_pct,avg_thumbs_up,sample_data_source
0,GPay,1_Early (2018-2019),8481,1.87,6312,74.400000000000,1384,16.300000000000,22.90,kaggle_help
1,GPay,2_Crisis (2020-2022),22945,1.57,19191,83.600000000000,1961,8.500000000000,32.60,kaggle_help
2,GPay,3_Recovery (2023-2026),33128,3.49,11226,33.900000000000,19871,60.000000000000,2.20,kaggle_new
3,Paytm,1_Early (2018-2019),8309,2.14,5667,68.200000000000,2038,24.500000000000,19.50,kaggle_help
4,Paytm,2_Crisis (2020-2022),15773,1.97,11519,73.000000000000,3266,20.700000000000,24.10,kaggle_help
5,Paytm,3_Recovery (2023-2026),38109,3.45,13625,35.800000000000,22971,60.300000000000,2.00,kaggle_new
6,PhonePe,1_Early (2018-2019),8297,2.23,5392,65.000000000000,2307,27.800000000000,11.90,kaggle_help
7,PhonePe,2_Crisis (2020-2022),19022,2.13,12762,67.100000000000,4563,24.000000000000,26.10,kaggle_help
8,PhonePe,3_Recovery (2023-2026),35570,3.93,7908,22.200000000000,25949,73.000000000000,2.60,kaggle_new



=== Q6 PART B: Complaint Breakdown by Era ===
all 7 categories per app per era including Other



,app,era,complaint_category,complaint_count,pct_of_era_negatives,rank_in_era
0,GPay,1_Early (2018-2019),Other,2358,37.400000000000,1
1,GPay,1_Early (2018-2019),Payment Failures,1379,21.800000000000,2
2,GPay,1_Early (2018-2019),Technical Failures,1233,19.500000000000,3
3,GPay,1_Early (2018-2019),Customer Support,593,9.400000000000,4
4,GPay,1_Early (2018-2019),UX Problems,413,6.500000000000,5
5,GPay,1_Early (2018-2019),Trust & Security,327,5.200000000000,6
6,GPay,1_Early (2018-2019),Regulatory Issues,9,0.100000000000,7
7,GPay,2_Crisis (2020-2022),Other,7615,39.700000000000,1
8,GPay,2_Crisis (2020-2022),Technical Failures,4310,22.500000000000,2
9,GPay,2_Crisis (2020-2022),Payment Failures,3506,18.300000000000,3



=== Q6 PART C: Executive Summary ===
9 rows — top specific complaint per app per era



,app,era,top_complaint,complaint_count,pct_of_negatives
0,GPay,1_Early (2018-2019),Payment Failures,1379,34.900000000000
1,GPay,2_Crisis (2020-2022),Technical Failures,4310,37.200000000000
2,GPay,3_Recovery (2023-2026),Technical Failures,1461,41.100000000000
3,Paytm,1_Early (2018-2019),Technical Failures,986,25.500000000000
4,Paytm,2_Crisis (2020-2022),Technical Failures,2015,26.900000000000
5,Paytm,3_Recovery (2023-2026),Technical Failures,2363,36.000000000000
6,PhonePe,1_Early (2018-2019),Payment Failures,1404,37.500000000000
7,PhonePe,2_Crisis (2020-2022),Payment Failures,2169,28.300000000000
8,PhonePe,3_Recovery (2023-2026),Technical Failures,1013,38.100000000000


## Step 10 - Q6 Part C Direct Fix

**Why this step exists:**
Due to encoding issues in the SQL file that affect how `run_sql_file`
splits batches, Q6 Part C is run directly here to guarantee the correct
result is always saved. This overwrites the CSV with the verified output.

This is the authoritative source for `q6_era_comparison_result3.csv`.

In [10]:
print('Running Q6 Part C directly...')
print('-' * 50)

with open('sql/query6_era_comparison.sql', 'r', encoding='utf-8') as f:
    content = f.read()

# Get the last batch (Part C)
batches = re.split(
    r'^\s*GO\s*$', content,
    flags=re.MULTILINE | re.IGNORECASE
)
batches = [b.strip() for b in batches if b.strip()]
last_batch = batches[-1]

try:
    cursor = conn.cursor()
    cursor.execute(last_batch)
    cols = [col[0] for col in cursor.description]
    rows = cursor.fetchall()
    q6c_fixed = pd.DataFrame.from_records(rows, columns=cols)

    # Save as the authoritative Part C result
    q6c_fixed.to_csv('sql_results/q6_era_comparison_result3.csv', index=False)

    print('Q6 Part C saved correctly.')
    print()
    print('=== Q6 PART C: Executive Summary (verified) ===')
    display(q6c_fixed[['app','era','top_complaint','pct_of_negatives']])

except Exception as e:
    print(f'Error: {e}')

Running Q6 Part C directly...
--------------------------------------------------
Q6 Part C saved correctly.

=== Q6 PART C: Executive Summary (verified) ===


,app,era,top_complaint,pct_of_negatives
0,GPay,1_Early (2018-2019),Payment Failures,34.900000000000
1,GPay,2_Crisis (2020-2022),Technical Failures,37.200000000000
2,GPay,3_Recovery (2023-2026),Technical Failures,41.100000000000
3,Paytm,1_Early (2018-2019),Technical Failures,25.500000000000
4,Paytm,2_Crisis (2020-2022),Technical Failures,26.900000000000
5,Paytm,3_Recovery (2023-2026),Technical Failures,36.000000000000
6,PhonePe,1_Early (2018-2019),Payment Failures,37.500000000000
7,PhonePe,2_Crisis (2020-2022),Payment Failures,28.300000000000
8,PhonePe,3_Recovery (2023-2026),Technical Failures,38.100000000000


## Step 11 - Verify All Exports
Confirm all 9 CSV files are present and correct.


In [11]:
print('=== EXPORT SUMMARY ===')
print()

expected = [
    ('q1_ratings_overview_result1.csv',    'Q1  Ratings overview per app'),
    ('q2_ratings_trend_result1.csv',        'Q2  Year by year trend'),
    ('q3_complaint_categories_result1.csv', 'Q3  Complaint categories'),
    ('q4_paytm_deepdive_result1.csv',       'Q4  Paytm monthly deep dive'),
    ('q5_response_analysis_result1.csv',    'Q5A Response rate overview'),
    ('q5_response_analysis_result2.csv',    'Q5B Response rate by year'),
    ('q6_era_comparison_result1.csv',       'Q6A Era sentiment'),
    ('q6_era_comparison_result2.csv',       'Q6B Era complaint breakdown'),
    ('q6_era_comparison_result3.csv',       'Q6C Executive summary'),
]

all_good = True
for filename, description in expected:
    path = f'sql_results/{filename}'
    if os.path.exists(path):
        df   = pd.read_csv(path)
        size = os.path.getsize(path) / 1024
        print(f'  [OK]  {filename:45s} {len(df):>4} rows  {size:.1f} KB')
        print(f'        {description}')
    else:
        print(f'  [MISSING] {filename}')
        all_good = False
    print()

if all_good:
    print('All 9 files present. Ready for Notebook 03.')
else:
    print('Some files missing. Re-run the relevant step.')

=== EXPORT SUMMARY ===

  [OK]  q1_ratings_overview_result1.csv                  3 rows  0.3 KB
        Q1  Ratings overview per app

  [OK]  q2_ratings_trend_result1.csv                    21 rows  0.9 KB
        Q2  Year by year trend

  [OK]  q3_complaint_categories_result1.csv             21 rows  1.0 KB
        Q3  Complaint categories

  [OK]  q4_paytm_deepdive_result1.csv                  120 rows  5.7 KB
        Q4  Paytm monthly deep dive

  [OK]  q5_response_analysis_result1.csv                 3 rows  0.4 KB
        Q5A Response rate overview

  [OK]  q5_response_analysis_result2.csv                18 rows  1.0 KB
        Q5B Response rate by year

  [OK]  q6_era_comparison_result1.csv                    9 rows  1.0 KB
        Q6A Era sentiment

  [OK]  q6_era_comparison_result2.csv                   63 rows  4.1 KB
        Q6B Era complaint breakdown

  [OK]  q6_era_comparison_result3.csv                    9 rows  0.7 KB
        Q6C Executive summary

All 9 files present. 

## Step 12 - Close Connection

In [12]:
conn.close()
print('SQL Server connection closed.')
print('Notebook 02 complete. Proceed to 03_visualisation.ipynb')

SQL Server connection closed.
Notebook 02 complete. Proceed to 03_visualisation.ipynb


---
## Notebook 02 Complete

| Step | Query | Result Sets | Key Finding |
|---|---|---|---|
| 4 | Q1 Ratings Overview | 1 | GPay worst at 56.9% negative |
| 5 | Q2 Year Trend | 1 | All apps bottomed 2020-22 |
| 6 | Q3 Complaint Categories | 1 | Paytm 20x more regulatory complaints |
| 7 | Q4 Paytm Deep Dive | 1 | 1,249% spike mid-2023 |
| 8 | Q5 Response Analysis | 2 | Paytm responds most, GPay least |
| 9 | Q6 Era Comparison | 3 | Payment failures fell, Technical persisted |
| 10 | Q6 Part C Fix | 1 | Executive summary without Other |

**Proceed to:** `03_visualisation.ipynb`
